# FastText finetuning notebook

This notebook is thought to be executed on google colab.

The notebook assumes a mounted google drive with the folder [HLT_artifacts_group_5](https://drive.google.com/drive/folders/1qjys2raxwIJMF7Dw8ss8mNQhCzYf9U7v?usp=drive_link) present

---

## Dependencies install

In [35]:
from google.colab import drive
drive.mount('/content/drive')

!pip install fasttext
!pip install numpy==1.24.4

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Global variables

This section declares global variables used for training and setting project paths

> If you're running this in a local environment you must set the paths accordingly

In [36]:
import fasttext
import csv
import os
import time
import math
from collections import defaultdict

# MODE = "multiclass"
MODE = "binary"

OVERRIDE = True # Rewrites all .txt datasets if true

ARTIFACTS = "/content/drive/MyDrive/HLT_artifacts_group_5"  # Google drive shared folder path
DATASET = f"{ARTIFACTS}/datasets"   # Path containing the test, validation, and training sets

# File names (without extension) of training, validation and test set
TRAINING = f"{DATASET}/train_{MODE}"
VALIDATION = f"{DATASET}/validation_{MODE}"
TEST = f"{DATASET}/test_{MODE}"

# path of the .vec pretrained vectors (extracted from crawl-300d-2M-subword.zip, downloadable from https://fasttext.cc/docs/en/english-vectors.html)
FAST_TEXT = f"{ARTIFACTS}/base_models/fasttext"
# path and filename in which the finetuned .bin model will be saved
FINE_TUNED = f"{ARTIFACTS}/fine-tuned_models/fasttext-{MODE}.bin"
TMP = "/content" # tmp folder used to store intermediate models


## Training and model parameters
PRETRAINED_DIM=300 # When ``ValueError: Dimension of pretrained vectors (x) does not match dimension (y)!`` set this var to x
BENCHMARK_TIME_LIMIT = 60 * 3 # Minimum time to benchmark the final model

EPOCHS = [18, 22] # Range of epochs used because the api doesn't allow for incremental training
PATIENCE = 2
LR = 1e-2

INPUT_COL = "tweet_text" # csv input column to use
OUT_COL = "label" # csv column containing ground truth

# Fine tuning

This section declares the functions used to fine tune the model and train a classifier on the Hate speech recognition task.

Among these measures only the Macro F1 score is used as a loss function for the validation step.

The others are used to show comparative results with other models.

In [37]:
def accuracy(cm: defaultdict) -> float:
    total = sum(cm.values())
    correct = sum(count for (true, pred), count in cm.items() if true == pred)
    return correct / total if total > 0 else 0.0

def macro_f1(cm: defaultdict) -> float:
    # Collect all unique labels from true and predicted
    labels = sorted(set([true for (true, _) in cm.keys()] + [pred for (_, pred) in cm.keys()]))
    f1s = []

    for label in labels:
        tp = cm.get((label, label), 0)
        fp = sum(cm.get((other, label), 0) for other in labels if other != label)
        fn = sum(cm.get((label, other), 0) for other in labels if other != label)

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

        f1s.append(f1)

    return sum(f1s) / len(f1s) if f1s else 0.0

def micro_f1(cm: defaultdict) -> float:
    tp = sum(count for (true, pred), count in cm.items() if true == pred)
    total = sum(cm.values())
    return tp / total if total > 0 else 0.0


The following class implemets methods to validate, fine-tune and benchmark the model.

In [38]:
class FineFastText:
    def __init__(self, model_path: str | None = None) -> None:
        self.model = None
        if model_path is not None:
          self.model = fasttext.load_model(model_path + ".bin")

    def validate(self, dataset = VALIDATION) -> dict:
      """
      Validation phase. Works for binary and multi-class.
      Returns:
          - loss (sum)
          - confusion matrix as a dict: cm[(true, pred)] = count
      """
      cm = defaultdict(int)
      n_samples = 0

      with open(f"{dataset}.txt", "r", encoding="utf-8") as file:
          for line in file:
              line = line.replace("\n", "")
              parts = line.strip().split(maxsplit=1)
              if len(parts) < 2:
                  continue

              true_label = parts[0].replace("__label__", "")
              text = parts[1]

              pred_label = self.model.predict(text, k=1)[0][0].replace("__label__", "")

              cm[(true_label, pred_label)] += 1
              n_samples += 1

      return {
          "cm": cm,
          "samples": n_samples,
          "loss": 1 - macro_f1(cm)
      }


    def fine_tune(self, epochs: list[int] = EPOCHS):
        best_model = ""
        best_val = {"loss": float('inf')}
        wait = 0
        for i in range(epochs[0], epochs[1]+1):
            # Train the model
            self.model = fasttext.train_supervised(input=f"{TRAINING}.txt", pretrainedVectors=FAST_TEXT + ".vec", dim=PRETRAINED_DIM, epoch=i, lr=LR)

            # Evaluate the model
            val = self.validate()
            print(f'Epoch {i} validation loss {val["loss"]}')
            if val["loss"] < best_val["loss"]:
              best_model = f"{TMP}/{i}.bin"
              self.model.save_model(best_model)
              best_val = val
              wait = 0
            if wait > PATIENCE:
              print(f"Early stopped at epoch {i}")
              break
            wait += 1

        # Compress the model
        self.model = fasttext.load_model(best_model)
        self.model.quantize(retrain=False)
        self.model.save_model(f"finetuned-{MODE}.ftq")
        # remove intermediate results
        for i in range(epochs[0], epochs[1]+1):
          path = f"{TMP}/{i}.bin"
          if os.path.exists(path):
            os.remove(path)


    def benchmark(self, dataset:str = TEST, model_path:str = FINE_TUNED) -> float:
        self.model = fasttext.load_model(model_path)

        start = time.time()
        i = 0
        while time.time() - start < BENCHMARK_TIME_LIMIT:
          self.model.test(f"{dataset}.txt")
          i += 1

        samples = 1
        with open(f"{dataset}.txt") as file:
          samples = sum(1 for _ in file)

        return i * samples, time.time() - start

The `DatasetAdapter` class implements methods for generating the dataset the `fasttext` API expects.

The original dataset, in csv format, has to be rewritten in txt and with all `\n` chars substituted or removed.

In [39]:
class DatasetAdapter:
    @staticmethod
    def prepare_dataset():
        assert os.path.exists(f"{TRAINING}.csv"), f"{TRAINING}.csv not found"
        assert os.path.exists(f"{VALIDATION}.csv") , f"{VALIDATION}.csv not found"
        assert os.path.exists(f"{TEST}.csv") , f"{TEST}.csv not found"

        for dataset in [TRAINING, VALIDATION, TEST]:
            DatasetAdapter._write_txt(dataset)

    @staticmethod
    def _write_txt(dataset: str):
        if os.path.exists(f"{dataset}.txt"):
            if not OVERRIDE:
              return
            os.remove(f"{dataset}.txt")

        txt = open(f"{dataset}.txt", "w", encoding="utf-8")
        file = open(f"{dataset}.csv", "r", encoding="utf-8")

        reader = csv.DictReader(file, delimiter=',')
        for row in reader:
          assert INPUT_COL in row , f"row of dataset {dataset} doesn't have {INPUT_COL} attribute, it cointains {row.keys()}"
          assert OUT_COL in row, f"row of dataset {dataset} doesn't have {OUT_COL} attribute, it cointains {row.keys()}"

          tweet = row[INPUT_COL]
          if "\n" in tweet:
            tweet = tweet.replace("\n", " _ENTER_ ")
          txt.write(f"__label__{row[OUT_COL]} {tweet}\n")

        file.close()
        txt.close()


DatasetAdapter.prepare_dataset()
print("Dataset ready")

Dataset ready


## Showing results

The following functions are used to print a confusion matrix

In [40]:
def binary_cm(metrics: dict):
  cm = metrics["cm"]
  print("Binary model: Confusion matrix")
  print("\t|Positive\t|Negative")
  print(f"True\t|{cm[('1', '1')]}\t\t|{cm[('1', '0')]}")
  print(f"False\t|{cm[('0', '1')]}\t\t|{cm[('0', '0')]}")

def multiclass_cm(metrics: dict):
    cm = metrics["cm"]

    labels = ['0', '1', '2', '3', '4']

    # Header row
    print("Multiclass Confusion Matrix")
    header = "\t|".join(labels)
    print(header)

    # Each row: true label
    for true_label in labels:
        row = []
        for pred_label in labels:
            count = cm.get((true_label, pred_label), 0)
            row.append(str(count))
        print("\t|".join(row))

### fine tuning

What follows are steps of finetuning, showing a confusion matrix and computing the previously mentioned measures.

In [41]:
# Fine tuning fasttext
fast = FineFastText()
print("\nFine tuning")
fast.fine_tune()

## Fine tuned model testing
metrics = fast.validate(dataset = TEST)
del fast


Fine tuning
Epoch 18 validation loss 0.12158968449176633
Epoch 19 validation loss 0.12138798376493454
Epoch 20 validation loss 0.12241883141354037
Epoch 21 validation loss 0.12277094075729189
Epoch 22 validation loss 0.12284643795919203
Early stopped at epoch 22


In [42]:
# Metrics and confusion matrix print
if MODE == "binary":
  binary_cm(metrics)
else:
  multiclass_cm(metrics)

Binary model: Confusion matrix
	|Positive	|Negative
True	|2989		|99
False	|215		|443


In [43]:
print("Accuracy:", accuracy(metrics["cm"]))
print("Macro F1:", macro_f1(metrics["cm"]))
print("Micro F1:", micro_f1(metrics["cm"]))

Accuracy: 0.9161772557394554
Macro F1: 0.8442143462598009
Micro F1: 0.9161772557394554


## Benchmarking

Given the whole point of using fasttext is achieving a good enough result without sacrificing performances makes sense to see how many samples it can calssify in a time slice.

In [44]:
fast = FineFastText()
samples, total_time = fast.benchmark(model_path=FINE_TUNED)
print(f"Benchmark finetuned, classified {samples} samples in {total_time}s.")
del fast

Benchmark finetuned, classified 7660570 samples in 180.10524559020996s.
